In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_H.xpt"
csv_file = DATA_DIR / "SMQFAM_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 SMQFAM_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

#
# IMPORTANT:
#
# SMQFAM_H differs from earlier SMQFAM cycles.
#
# 2013-2014 contains:
#
#     SEQN
#     SMD460
#     SMD470
#     SMD480
#

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD460",
    "SMD470",
    "SMD480"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_H columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 10175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 4:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,175 rows x 4 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# All four fields are whole-number variables.
#
#
# SEQN:
#     Respondent sequence number.
#
# SMD460:
#     Number of household members who smoke tobacco.
#
# SMD470:
#     Number of household members who smoke
#     inside the home.
#
# SMD480:
#     Number of days during past 7 days someone
#     smoked tobacco inside the home.
#
#
# There are:
#
#     NO genuine decimals
#     NO text/string variables
#
# Therefore all 4 can ultimately become nullable Int64.
#
#
# IMPORTANT:
#
# This applies ONLY to SMQFAM_H.
#
# Genuine Dietary / Laboratory / Examination /
# sample-weight decimals in other datasets remain
# exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 4:

    raise ValueError(
        "Expected exactly 4 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value without validating
# exactly where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all SMQFAM_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached SMQFAM_H.xpt:
#
# SMD460 = 7,601
# SMD470 = 1,145
# SMD480 =    98
#
# TOTAL  = 8,844
#
#
# CDC confirms:
#
# SMD460 = 0
#     No one in household is a smoker
#     n = 7,601
#
# SMD470 = 0
#     No one smokes inside house
#     n = 1,145
#
# SMD480 = 0
#     0 days someone smoked inside
#     n = 98
#

EXPECTED_TINY_COUNTS = {
    "SMD460": 7601,
    "SMD470": 1145,
    "SMD480": 98
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated SMQFAM_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 8844:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros already existed "
        "before correction."
    )


print(
    "\nPASS: Exact 8,844-value tiny-zero "
    "pattern across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Repair only:
#
#     SMD460
#     SMD470
#     SMD480
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 8844:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents EXACTLY what the corrected
# XPT values should be.
#
# The ONLY permitted changes are:
#
#     validated tiny -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT values remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a test.
#
# It does NOT modify any data.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(
            values
        ),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in SMQFAM_H. They will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(
        seqn_values
    ),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 12. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All four fields have now been proven to contain
# whole-number values only.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(
                values
            ),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 4 SMQFAM_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC exactly."
    )


# ============================================================
# 14. CDC CHECK - SMD460
# ============================================================

#
# Number of people in household who smoke tobacco.
#
# CDC:
#
# 0   No one                         7,601
# 1   1 household smoker            1,403
# 2   2 household smokers             849
# 3   3 or more                       200
# 777 Refused                           5
# 999 Don't know                        1
# Missing                             116
#

check_frequency(
    "SMD460",
    {
        0: 7601,
        1: 1403,
        2: 849,
        3: 200,
        777: 5,
        999: 1
    },
    116
)


# ============================================================
# 15. CDC CHECK - SMD470
# ============================================================

#
# Number of people who smoke INSIDE home.
#
# CDC:
#
# 0   No one                       1,145
# 1   1 person                       724
# 2   2 people                       511
# 3   3 or more                       72
# Missing                          7,723
#

check_frequency(
    "SMD470",
    {
        0: 1145,
        1: 724,
        2: 511,
        3: 72
    },
    7723
)


# ============================================================
# 16. SMD460 -> SMD470 ROUTING CHECK
# ============================================================

#
# CDC questionnaire:
#
# If SMD460 is:
#
#     0
#     Refused
#     Don't know
#
# respondent skips to end of section.
#
# Therefore SMD470 should be populated exactly when:
#
#     SMD460 in {1,2,3}
#

smd470_expected = (
    df["SMD460"]
    .isin(
        [
            1,
            2,
            3
        ]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smd470_present = (
    df["SMD470"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smd470_route_mismatch = int(
    np.sum(
        smd470_expected
        !=
        smd470_present
    )
)


print(
    "\n--- SMD460 / SMD470 ROUTING ---"
)


print(
    "Routing mismatches:",
    smd470_route_mismatch
)


if smd470_route_mismatch != 0:

    raise ValueError(
        "SMD470 routing does not match "
        "SMD460 household-smoker responses."
    )


print(
    "PASS: SMD470 routing validated exactly."
)


# ============================================================
# 17. SMD470 <= SMD460 CONSISTENCY CHECK
# ============================================================

#
# CDC applies a hard edit:
#
# Number who smoke inside cannot exceed number of
# household smokers.
#

valid_pair = (
    df["SMD460"]
    .isin(
        [
            1,
            2,
            3
        ]
    )
    &
    df["SMD470"]
    .notna()
)


violations = int(
    (
        valid_pair
        &
        (
            df["SMD470"]
            >
            df["SMD460"]
        )
    )
    .sum()
)


print(
    "\n--- HOUSEHOLD SMOKER CONSISTENCY ---"
)


print(
    "SMD470 > SMD460 violations:",
    violations
)


if violations != 0:

    raise ValueError(
        "SMD470 exceeds SMD460 in some records."
    )


print(
    "PASS: SMD470 never exceeds SMD460."
)


# ============================================================
# 18. CDC CHECK - SMD480
# ============================================================

#
# Number of days during past 7 days someone smoked
# tobacco inside the home.
#
# CDC:
#
# 0 = 98
# 1 = 179
# 2 = 230
# 3 = 49
# 4 = 35
# 5 = 27
# 6 = 7
# 7 = 679
# 99 = Don't know = 3
#
# Missing = 8,868
#

check_frequency(
    "SMD480",
    {
        0: 98,
        1: 179,
        2: 230,
        3: 49,
        4: 35,
        5: 27,
        6: 7,
        7: 679,
        99: 3
    },
    8868
)


# ============================================================
# 19. SMD470 -> SMD480 ROUTING CHECK
# ============================================================

#
# If SMD470 = 0, questionnaire ends.
#
# SMD480 is asked when at least one household member
# smokes inside:
#
#     SMD470 in {1,2,3}
#

smd480_expected = (
    df["SMD470"]
    .isin(
        [
            1,
            2,
            3
        ]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smd480_present = (
    df["SMD480"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smd480_route_mismatch = int(
    np.sum(
        smd480_expected
        !=
        smd480_present
    )
)


print(
    "\n--- SMD470 / SMD480 ROUTING ---"
)


print(
    "Routing mismatches:",
    smd480_route_mismatch
)


if smd480_route_mismatch != 0:

    raise ValueError(
        "SMD480 routing does not match "
        "SMD470 indoor-smoker responses."
    )


print(
    "PASS: SMD480 routing validated exactly."
)


# ============================================================
# 20. FINAL ZERO PATTERN
# ============================================================

#
# Original pandas-decoded source contained:
#
#     ordinary zeros = 0
#
# After correction the ONLY zeros must be:
#
# SMD460 = 7,601
# SMD470 = 1,145
# SMD480 =    98
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from validated "
        "SMQFAM_H pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 8844:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 8,844 validated restorations."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numeric changes are:
#
#     SMD460 tiny -> 0
#     SMD470 tiny -> 0
#     SMD480 tiny -> 0
#
# Everything else must remain exactly identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            fidelity_failures
        )
    )


print(
    "PASS: All 4 variables match the exact "
    "corrected XPT values."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
#
# SMQFAM_H contains no genuine decimal fields.
#
# The same export convention remains safe for your
# broader NHANES workflow where Dietary/Lab/Exam
# genuine decimal values must remain AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 23. RE-READ CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 24. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: All 4 CSV variables round-trip "
    "to exact corrected XPT values."
)


# ============================================================
# 25. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 26. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 27. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 28. RAW CSV .0 CHECK
# ============================================================

#
# SMQFAM_H has NO genuine decimal variables.
#
# Therefore checking all 4 fields for unnecessary ".0"
# is safe for THIS file.
#
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# SMD460
#     tiny -> 0
#     1.0  -> 1
#
# SMD470
#     tiny -> 0
#
# SMD480
#     tiny -> 0
#     7.0  -> 7
#
#
# DO NOT use this global method on datasets containing
# genuine decimal fields.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any SMQFAM_H field."
)


# ============================================================
# 29. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches all 8,844 validated restorations."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "SMQFAM_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - "
    "Smoking / Household Smokers"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "SMD460 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD460"]
)


print(
    "SMD470 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD470"]
)


print(
    "SMD480 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD480"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "String variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Households with no smoker:",
    int(
        df["SMD460"]
        .eq(0)
        .sum()
    )
)


print(
    "Households with 1 smoker:",
    int(
        df["SMD460"]
        .eq(1)
        .sum()
    )
)


print(
    "No one smokes inside home:",
    int(
        df["SMD470"]
        .eq(0)
        .sum()
    )
)


print(
    "Indoor smoking 0 days past week:",
    int(
        df["SMD480"]
        .eq(0)
        .sum()
    )
)


print(
    "Indoor smoking 7 days past week:",
    int(
        df["SMD480"]
        .eq(7)
        .sum()
    )
)


print(
    "XPT -> CSV numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 SMQFAM_H XPT -> CSV
Rows: 10,175
Columns: 4
PASS: Structure = 10,175 rows x 4 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number fields: ['SEQN', 'SMD460', 'SMD470', 'SMD480']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 8,844

Tiny values by variable:
SMD460    7601
SMD470    1145
SMD480      98

PASS: Exact 8,844-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
SMD460: 7,601 restored to 0
SMD470: 1,145 restored to 0
SMD480: 98 restored to 0

PASS: 8,844 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
SMD460: 0 fractional observations
SMD470: 0 fractional observations
SMD480: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 73557
Maximum: 83731
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83731 